# 1.2 Tools

Models are good at language, not at exact arithmetic or talking to APIs. A **tool** is a Python function the agent can call: you decorate it, describe it, and pass it into `create_agent`.

This notebook defines a square-root tool three equivalent ways, invokes it directly, then attaches it to an agent and inspects `tool_calls` in the message list.

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Tool definition

The `@tool` decorator turns a function into a LangChain tool. The model uses:

- **name** — function name by default, or the first decorator argument
- **description** — the docstring, or an explicit `description=`
- **arguments** — from type hints (here `x: float`)

The three cells below are the same tool, written with more of that metadata moved onto the decorator.

In [ ]:
from langchain.tools import tool

# Name = "square_root", description = docstring
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
# Explicit tool name; function can be named anything (tool1)
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
# Name + description on the decorator — no docstring required
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

You can call a tool **without** an agent. Arguments are a dict of parameter names. This is useful for unit-testing the function before wiring it up.

In [ ]:
# Direct call: same signature the model will use
tool1.invoke({"x": 467})

## Adding tools to agents

Pass `tools=[tool1]` and a system prompt that tells the model **when** to use them. The agent loop: model may emit a tool call → runtime runs the function → result goes back as a tool message → model writes the final answer.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[tool1],
    # Prompt mentions square as well; this agent only has square_root — that's OK for the demo
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

# Final natural-language answer (after any tool round-trips)
print(response['messages'][-1].content)

### Inspect the message trace

Pretty-print the full list: human question, AI message with `tool_calls`, tool result, then the closing AI message. `messages[1].tool_calls` is the structured call the model requested (name + args).

In [ ]:
from pprint import pprint

pprint(response['messages'])

In [ ]:
# First AI turn should include a call to square_root with x=467
print(response["messages"][1].tool_calls)